In [ ]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score, fbeta_score, make_scorer
from imblearn.ensemble import BalancedBaggingClassifier, BalancedRandomForestClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

# Define features and target variable

In [3]:
# Define features and target variable
features = [
    # encoded station/service
    'StationCode_TE',
    'Service:Type_Intercity',
    'Service:Type_Intercity direct',
    'Service:Type_Sprinter',
    # temporal
    'Hour_sin',
    'Hour_cos',
    'DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'IsWeekend',
    'RushHour',
    # operational context
    'StationTraffic',
    'Stop:Platform change',
    'arrival_scheduled',
    'departure_scheduled',
    # weather
    'Wind Direction',
    'Hourly Mean Wind Speed',
    'Wind Speed last 10 Minutes',
    'Max Wind Speed',
    'Temperature',
    'Dew Point temperature',
    'Sunshine Duration',
    'Global Radiation',
    'Precipitation Duration',
    'Precipitation Amount',
    'Air Pressure',
    'Horizontal Visibility',
    'Cloud Cover',
    'Humidity',
    'Fog',
    'Rainfall',
    'Snowfall',
    'Thunder',
    'Hail',
]

target = "is_cancelled"

# Define chunck reader function

In [4]:
# Prepare for chunked reading
cols = features + [target]
chunk_size = 1_000_000
sample_size = 5_000_000
random_state = 42

dtype_map = {col: "float32" for col in features}
dtype_map[target] = "int8"


# Read CSV in chunks
def chunk_reader(file_path):
    for chunk in pd.read_csv(
        file_path,
        usecols=cols,
        dtype=dtype_map,
        chunksize=chunk_size
    ):
        chunk = chunk.reindex(columns=cols, fill_value=0) # Ensure all columns are present

        X_chunk = chunk[features]
        y_chunk = chunk[target]

        yield X_chunk, y_chunk # Yield the chunk for processing
        
# Count classes
not_cancelled_total = 0
cancelled_total = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):
    not_cancelled_total += (y_chunk == 0).sum()
    cancelled_total += (y_chunk == 1).sum()

total_rows = not_cancelled_total + cancelled_total

print(f"Train not cancelled: {not_cancelled_total:,}")
print(f"Train cancelled: {cancelled_total:,}")
print(f"Train total:     {total_rows:,}")

# Decide sample sizes
not_cancelled_sample_size = int(sample_size * not_cancelled_total / total_rows)
cancelled_sample_size = sample_size - not_cancelled_sample_size

print(f"Sampling not cancelled: {not_cancelled_sample_size:,}")
print(f"Sampling cancelled: {cancelled_sample_size:,}")

Train not cancelled: 43,903,090
Train cancelled: 4,479,366
Train total:     48,382,456
Sampling not cancelled: 907,417
Sampling cancelled: 92,583


# Define cancelled/not-cancelled

In [5]:
# Generate random numbers for sampling (default_rng is faster)
rng = np.random.default_rng(random_state)

X_not_cancelled_parts = []
y_not_cancelled_parts = []
X_cancelled_parts = []
y_cancelled_parts = []

not_cancelled_seen = 0
cancelled_seen = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):

    # Create masks for cancelled and not cancelled
    not_cancelled_mask = y_chunk == 0
    cancelled_mask = y_chunk == 1
    
    # Split the chunk into cancelled and not cancelled
    X_not_cancelled = X_chunk.loc[not_cancelled_mask]
    y_not_cancelled = y_chunk.loc[not_cancelled_mask]

    X_cancelled = X_chunk.loc[cancelled_mask]
    y_cancelled = y_chunk.loc[cancelled_mask]

    # sample fraction based on remaining needed / remaining available
    neg_remaining_needed = not_cancelled_sample_size - sum(len(p) for p in y_not_cancelled_parts)
    pos_remaining_needed = cancelled_sample_size - sum(len(p) for p in y_cancelled_parts)

    neg_remaining_total = not_cancelled_total - not_cancelled_seen
    pos_remaining_total = cancelled_total - cancelled_seen

    # Sample not cancelled
    if neg_remaining_needed > 0 and len(X_not_cancelled) > 0:
        p_neg = min(1.0, neg_remaining_needed / max(neg_remaining_total, 1))
        keep_neg = rng.random(len(X_not_cancelled)) < p_neg

        X_not_cancelled_parts.append(X_not_cancelled.loc[keep_neg])
        y_not_cancelled_parts.append(y_not_cancelled.loc[keep_neg])
        
    # Sample cancelled
    if pos_remaining_needed > 0 and len(X_cancelled) > 0:
        p_pos = min(1.0, pos_remaining_needed / max(pos_remaining_total, 1))
        keep_pos = rng.random(len(X_cancelled)) < p_pos

        X_cancelled_parts.append(X_cancelled.loc[keep_pos])
        y_cancelled_parts.append(y_cancelled.loc[keep_pos])

    not_cancelled_seen += len(X_not_cancelled)
    cancelled_seen += len(X_cancelled)

# Combine sampled parts
X_train_sample = pd.concat(X_not_cancelled_parts + X_cancelled_parts, axis=0)
y_train_sample = pd.concat(y_not_cancelled_parts + y_cancelled_parts, axis=0)

# Shuffle final sample
shuffle_idx = rng.permutation(len(X_train_sample))
X_train_sample = X_train_sample.iloc[shuffle_idx].to_numpy(dtype=np.float32, copy=False)
y_train_sample = y_train_sample.iloc[shuffle_idx].to_numpy(dtype=np.int8, copy=False)

# Define Evaluation Method

In [6]:
# Evaluate CSV split in chunks with custom threshold
def evaluate_split_xgb(name, file_path, model, threshold):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        y_prob = model.predict_proba(X_chunk_np)[:, 1]
        y_pred = (y_prob >= threshold).astype(int)

        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(y_pred)
        y_prob_parts.append(y_prob)

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    f2 = fbeta_score(y_true_all, y_pred_all, beta=2)
    pr_auc = average_precision_score(y_true_all, y_prob_all)

    print(f"\n{name}")
    print("Threshold:", round(threshold, 2))
    print("F2 Score:", round(f2, 4))
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("PR-AUC:", round(pr_auc, 4))

    return {
        "threshold": threshold,
        "f2": f2,
        "pr_auc": pr_auc
    }


# Find best threshold using validation data
def find_best_threshold_xgb(file_path, model):
    y_true_parts = []
    y_prob_parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        y_prob = model.predict_proba(X_chunk_np)[:, 1]

        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_prob_parts.append(y_prob)

    y_true_all = np.concatenate(y_true_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    best_threshold = 0.5
    best_f2 = -1

    for threshold in np.arange(0.01, 0.99, 0.01):
        y_pred_all = (y_prob_all >= threshold).astype(int)
        f2 = fbeta_score(y_true_all, y_pred_all, beta=2)

        if f2 > best_f2:
            best_f2 = f2
            best_threshold = threshold

    print("Best validation threshold:", round(best_threshold, 2))
    print("Best validation F2:", round(best_f2, 4))

    return best_threshold

# Finding the best parameters

In [7]:
# Custom F2 scorer
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

# Parameter grid for imbalance
param_dist = {
    "n_estimators": [50, 100, 200, 300, 500],
    "max_depth": [10, 15, 20, 50, None],
    "min_samples_split": [5, 10, 20, 50, 100],
    "min_samples_leaf": [2, 5, 10, 20, 50],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "class_weight": [
        "balanced",
        "balanced_subsample",
        {0: 1, 1: 5},
        {0: 1, 1: 10},
        {0: 1, 1: 15},
        {0: 1, 1: 20}
    ],
    "criterion": ["gini", "entropy", "log_loss"]
}

# Use smaller subset for tuning
X_tune, X_val_tune, y_tune, y_val_tune = train_test_split(
    X_train_sample,
    y_train_sample,
    train_size=0.3,
    stratify=y_train_sample,
    random_state=42
)

# Stratified 5-fold CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Base Random Forest
rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# Random search
rf_tuned = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=50,
    scoring=f2_scorer,
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_tune, y_tune)

print(f"Best CV F2 score: {rf_tuned.best_score_:.4f}")
print("Best parameters:")
print(rf_tuned.best_params_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


KeyboardInterrupt: 

# Random Forest: Balanced Bagging

In [ ]:
# Final model training with best parameters
bbc = BalancedBaggingClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=0.3,
    sampling_strategy="auto",
    replacement=True,
    class_weight=None,
    criterion="entropy",
    random_state=42,
    n_jobs=-1
)

bbc.fit(X_train_sample, y_train_sample)

print("Balanced Bagging Classifier trained on", len(y_train_sample), "samples")

best_threshold = find_best_threshold_rf("val_data.csv", bbc)

train_results = evaluate_split_rf(
    name="Train",
    file_path="train_data.csv",
    model=bbc,
    threshold=best_threshold
)

val_results = evaluate_split_rf(
    name="Validation",
    file_path="val_data.csv",
    model=bbc,
    threshold=best_threshold
)

test_results = evaluate_split_rf(
    name="Test",
    file_path="test_data.csv",
    model=bbc,
    threshold=best_threshold
)

# Random Forest: Balanced Random Forest Classifier

In [ ]:
# Final model training with best parameters
brf = BalancedRandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=0.3,
    sampling_strategy="auto",
    replacement=True,
    class_weight=None,
    criterion="entropy",
    random_state=42,
    n_jobs=-1
)

brf.fit(X_train_sample, y_train_sample)

print(f"Balanced Random Forest trained on {len(y_train_sample):,} samples")

best_threshold = find_best_threshold_rf("val_data.csv", brf)

train_results = evaluate_split_rf(
    "Train",
    "train_data.csv",
    brf,
    best_threshold
)

val_results = evaluate_split_rf(
    "Validation",
    "val_data.csv",
    brf,
    best_threshold
)

test_results = evaluate_split_rf(
    "Test",
    "test_data.csv",
    brf,
    best_threshold
)

# Random Forest: RandomUnderSampling 

In [21]:
# Random undersampling to balance classes
print("Before undersampling:", Counter(y_train_sample))

rus = RandomUnderSampler(
    sampling_strategy=0.25,
    random_state=42
)

X_train_under, y_train_under = rus.fit_resample(
    X_train_sample,
    y_train_sample
)

print("After undersampling:", Counter(y_train_under))

rf_rus = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features=0.3,
    class_weight={0: 1, 1: 5},
    criterion='entropy',
    random_state=42,
    n_jobs=-1
)

rf_rus.fit(X_train_under, y_train_under)

print(f"Random Forest trained on {len(y_train_under):,} samples")

best_threshold = find_best_threshold_xgb("val_data.csv", rf_rus)

train_results = evaluate_split_xgb(
    name="Train",
    file_path="train_data.csv",
    model=rf_rus,
    threshold=best_threshold
)

val_results = evaluate_split_xgb(
    name="Validation",
    file_path="val_data.csv",
    model=rf_rus,
    threshold=best_threshold
)

test_results = evaluate_split_xgb(
    name="Test",
    file_path="test_data.csv",
    model=rf_rus,
    threshold=best_threshold
)

Before undersampling: Counter({np.int8(0): 907007, np.int8(1): 92794})
After undersampling: Counter({np.int8(0): 371176, np.int8(1): 92794})
Random Forest trained on 463,970 samples


# Random Forest: RandomOverSampler

In [ ]:
# Random oversampling to balance classes
print("Before oversampling:", Counter(y_train_sample))

ros = RandomOverSampler(
    sampling_strategy=0.25,
    random_state=42
)

X_train_over, y_train_over = ros.fit_resample(
    X_train_sample,
    y_train_sample
)

print("After oversampling:", Counter(y_train_over))

rf_ros = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features=0.3,
    class_weight={0: 1, 1: 5},
    criterion='entropy',
    random_state=42,
    n_jobs=-1
)

rf_ros.fit(X_train_over, y_train_over)

print(f"Random Forest trained on {len(y_train_over):,} samples")

best_threshold = find_best_threshold_xgb("val_data.csv", rf_ros)

train_results = evaluate_split_xgb(
    name="Train",
    file_path="train_data.csv",
    model=rf_ros,
    threshold=best_threshold
)

val_results = evaluate_split_xgb(
    name="Validation",
    file_path="val_data.csv",
    model=rf_ros,
    threshold=best_threshold
)

test_results = evaluate_split_xgb(
    name="Test",
    file_path="test_data.csv",
    model=rf_ros,
    threshold=best_threshold
)

# Feature Importance

In [ ]:
# Feature importance analysis with best model - XGBoost: Class weighting (xgb_class)
importance_gain = xgb_class.get_booster().get_score(importance_type="gain")

importance_df = pd.DataFrame({
    "feature_index": importance_gain.keys(),
    "gain": importance_gain.values()
})

importance_df["feature"] = importance_df["feature_index"].apply(
    lambda x: features[int(x[1:])]
)

importance_df = importance_df[["feature", "gain"]].sort_values(
    "gain",
    ascending=False
)

print(importance_df.head(20))

                          feature         gain
3           Service:Type_Sprinter  1090.592041
12           Stop:Platform change   162.499725
14            departure_scheduled   103.930511
13              arrival_scheduled   102.664505
2   Service:Type_Intercity direct    83.070572
8                       Month_sin    62.024502
9                       Month_cos    56.827473
0                  StationCode_TE    53.788254
6                   DayOfWeek_sin    51.689621
16         Hourly Mean Wind Speed    48.633350
25                   Air Pressure    48.323803
18                 Max Wind Speed    48.018520
7                   DayOfWeek_cos    47.956501
5                        Hour_cos    47.794033
31                       Snowfall    44.879059
20          Dew Point temperature    43.846596
19                    Temperature    43.681908
24           Precipitation Amount    43.506943
23         Precipitation Duration    42.496460
15                 Wind Direction    41.988422


In [ ]:
# Create validation set for permutation importance

X_val_perm, y_val_perm = next(chunk_reader("val_data.csv"))

X_val_perm = X_val_perm.to_numpy(dtype=np.float32, copy=False)
y_val_perm = y_val_perm.to_numpy(copy=False)

print("Permutation validation sample shape:", X_val_perm.shape)
print("Permutation validation target shape:", y_val_perm.shape)

best_threshold = 0.25

def f2_threshold_scorer(estimator, X, y):
    y_prob = estimator.predict_proba(X)[:, 1]
    y_pred = (y_prob >= best_threshold).astype(int)
    return fbeta_score(y, y_pred, beta=2)

perm = permutation_importance(
    xgb_class,
    X_val_perm,
    y_val_perm,
    scoring=f2_threshold_scorer,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print("\nTop 20 permutation importances:")
print(perm_df.head(20))

print("\nLowest 10 permutation importances:")
print(perm_df.tail(10))

Permutation validation sample shape: (1000000, 35)
Permutation validation target shape: (1000000,)

Top 20 permutation importances:
                          feature  importance_mean  importance_std
0                  StationCode_TE     7.586199e-02    6.120673e-04
3           Service:Type_Sprinter     6.252007e-02    5.847214e-04
13           Stop:Platform change     8.474886e-03    1.345022e-04
15            departure_scheduled     6.203962e-03    1.609110e-04
12                 StationTraffic     5.679633e-03    1.371386e-04
14              arrival_scheduled     5.121368e-03    8.699289e-05
27          Horizontal Visibility     4.421139e-03    1.871901e-04
5                        Hour_cos     4.316275e-03    4.137232e-04
7                   DayOfWeek_cos     3.609400e-03    2.883970e-04
1          Service:Type_Intercity     2.494785e-03    1.032928e-04
16                 Wind Direction     2.185278e-03    4.028518e-04
20                    Temperature     1.683708e-03    3.660724e-

# Error Analysis

In [ ]:
# Get predictions for test set in a DataFrame
def get_predictions_dataframe(file_path, model, threshold):
    parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        y_prob = model.predict_proba(X_chunk_np)[:, 1]
        y_pred = (y_prob >= threshold).astype(int)

        chunk_results = X_chunk.copy()
        chunk_results["y_true"] = y_chunk.to_numpy(copy=False)
        chunk_results["y_prob"] = y_prob
        chunk_results["y_pred"] = y_pred

        parts.append(chunk_results)

    return pd.concat(parts, axis=0, ignore_index=True)


test_pred_df = get_predictions_dataframe(
    file_path="test_data.csv",
    model=xgb_class,
    threshold=0.25
)

# If too large, use:
# test_pred_df = test_pred_df.sample(
#     n=1_000_000,
#     random_state=42
# )

# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(
    test_pred_df["y_true"],
    test_pred_df["y_pred"]
)

print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not cancelled", "Cancelled"]
)

disp.plot(values_format=",d")
plt.title("Confusion Matrix - Final XGBoost Model")
plt.show()

In [ ]:
# calculate error types:

test_pred_df["error_type"] = np.select(
    [
        (test_pred_df["y_true"] == 0) & (test_pred_df["y_pred"] == 0),
        (test_pred_df["y_true"] == 0) & (test_pred_df["y_pred"] == 1),
        (test_pred_df["y_true"] == 1) & (test_pred_df["y_pred"] == 0),
        (test_pred_df["y_true"] == 1) & (test_pred_df["y_pred"] == 1),
    ],
    [
        "True Negative",
        "False Positive",
        "False Negative",
        "True Positive",
    ]
)

print(test_pred_df["error_type"].value_counts())

# Subgroup bias analysis

In [ ]:
from sklearn.metrics import precision_score, recall_score, fbeta_score

# Subgroups are:
# StationCode_TE
# Service type
# Hour / RushHour
# IsWeekend
# StationTraffic

def subgroup_metrics(df, group_col):
    rows = []

    for group_value, group_df in df.groupby(group_col):
        if len(group_df) < 1000:
            continue

        y_true = group_df["y_true"]
        y_pred = group_df["y_pred"]

        rows.append({
            "group": group_value,
            "n": len(group_df),
            "actual_cancel_rate": y_true.mean(),
            "predicted_cancel_rate": y_pred.mean(),
            "precision_cancelled": precision_score(y_true, y_pred, zero_division=0),
            "recall_cancelled": recall_score(y_true, y_pred, zero_division=0),
            "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        })

    return pd.DataFrame(rows).sort_values("f2", ascending=True)


print(subgroup_metrics(test_pred_df, "Service:Type_Intercity"))
print(subgroup_metrics(test_pred_df, "Service:Type_Sprinter"))
print(subgroup_metrics(test_pred_df, "IsWeekend"))
print(subgroup_metrics(test_pred_df, "RushHour"))

# For one-hot service type, create one readable service column first:
def get_service_type(row):
    if row["Service:Type_Intercity"] == 1:
        return "Intercity"
    elif row["Service:Type_Intercity direct"] == 1:
        return "Intercity direct"
    elif row["Service:Type_Sprinter"] == 1:
        return "Sprinter"
    else:
        return "Other"

test_pred_df["service_type"] = test_pred_df.apply(get_service_type, axis=1)

service_bias = subgroup_metrics(test_pred_df, "service_type")
print(service_bias)

# Rare Weather Conditions analysis

In [ ]:
weather_flags = ["Fog", "Rainfall", "Snowfall", "Thunder", "Hail"]

weather_results = []

for weather_col in weather_flags:
    for value, group_df in test_pred_df.groupby(weather_col):
        if len(group_df) < 1000:
            continue

        y_true = group_df["y_true"]
        y_pred = group_df["y_pred"]

        weather_results.append({
            "weather_feature": weather_col,
            "condition_present": value,
            "n": len(group_df),
            "actual_cancel_rate": y_true.mean(),
            "predicted_cancel_rate": y_pred.mean(),
            "precision_cancelled": precision_score(y_true, y_pred, zero_division=0),
            "recall_cancelled": recall_score(y_true, y_pred, zero_division=0),
            "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        })

weather_results_df = pd.DataFrame(weather_results).sort_values(
    ["weather_feature", "condition_present"]
)

print(weather_results_df)